In [11]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import torch.mps as mps
import os
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
from PIL import Image
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import time
import random

image_path = '../train_test_images'
classified_images = '../classified_images'
all_images = '../camera_frames'
model_path = 'model.pth'

In [14]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(device)

mps


In [3]:
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, shear=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # mean?
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(os.path.join(image_path, 'train'), transform=train_transforms)
val_data = datasets.ImageFolder(os.path.join(image_path, 'val'), transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

class_names = train_loader.dataset.classes  # Assuming ImageFolder dataset

In [4]:
def setup_model():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, len(class_names))
    model = model.to(device)
    return model

In [5]:
def train_model(model, train_loader, val_loader, epochs, scheduler=None, grad_clip=None, save_path=model_path):

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss()

    best_val_acc = 0.0  # Track the best validation accuracy

    for epoch in range(epochs):
        start_time = time.time()

        # Training
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        running_precision, running_recall, running_f1, running_auc = 0.0, 0.0, 0.0, 0.0
        total_grad_norm = 0.0
        all_labels = []
        all_preds = []

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()

            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            precision = (predicted == labels).float().mean().item()
            recall = correct / total
            running_precision += precision
            running_recall += recall

            grad_norm = sum(p.grad.data.norm(2).item() ** 2 for p in model.parameters() if p.grad is not None) ** 0.5
            total_grad_norm += grad_norm

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        epoch_precision = running_precision / len(train_loader)
        epoch_recall = running_recall / len(train_loader)
        avg_grad_norm = total_grad_norm / len(train_loader)

        epoch_f1 = f1_score(all_labels, all_preds, average='macro')
        cm = confusion_matrix(all_labels, all_preds)
        try:
            epoch_roc_auc = roc_auc_score(all_labels, F.softmax(outputs, dim=1).cpu().detach().numpy(), multi_class='ovr')
        except ValueError:
            epoch_roc_auc = float('nan')

        weight_norm = sum(p.data.norm(2).item() ** 2 for p in model.parameters()) ** 0.5

        if scheduler:
            scheduler.step()

        epoch_time = time.time() - start_time

        print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, '
              f'Precision: {epoch_precision:.4f}, Recall: {epoch_recall:.4f}, F1: {epoch_f1:.4f}, '
              f'AUC-ROC: {epoch_roc_auc:.4f}, Grad Norm: {avg_grad_norm:.4f}, Weight Norm: {weight_norm:.4f}, '
              f'Time: {epoch_time:.2f}s')

        print(f'Confusion Matrix:\n{cm}')

        # Validation
        model.eval()
        val_loss = 0.0
        correct_val, total_val = 0, 0
        val_precision, val_recall, val_f1, val_auc = 0.0, 0.0, 0.0, 0.0
        all_val_labels = []
        all_val_preds = []

        with torch.no_grad():
            for batch_idx, (inputs, labels) in enumerate(val_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

                val_precision += (predicted == labels).float().mean().item()
                val_recall += correct_val / total_val

        val_loss /= len(val_loader)
        val_acc = correct_val / total_val
        val_precision /= len(val_loader)
        val_recall /= len(val_loader)

        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro')
        val_cm = confusion_matrix(all_val_labels, all_val_preds)
        try:
            val_auc = roc_auc_score(all_val_labels, F.softmax(outputs, dim=1).cpu().numpy(), multi_class='ovr')
        except ValueError:
            val_auc = float('nan')

        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}, '
              f'Validation Precision: {val_precision:.4f}, Validation Recall: {val_recall:.4f}, '
              f'Validation F1: {val_f1:.4f}, Validation AUC-ROC: {val_auc:.4f}')
        print(f'Validation Confusion Matrix:\n{val_cm}')


        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f'Model saved at {save_path}')

In [7]:
train_model(setup_model(), train_loader, val_loader, epochs=10)

Epoch [1/10], Loss: 0.3078, Accuracy: 0.8899, Precision: 0.8895, Recall: 0.8414, F1: 0.8722, AUC-ROC: nan, Grad Norm: 2.7951, Weight Norm: 87.8792, Time: 130.13s
Confusion Matrix:
[[ 548    9  132]
 [   1 1698   83]
 [  96   99 1148]]
Validation Loss: 0.4441, Validation Accuracy: 0.8450, Validation Precision: 0.8456, Validation Recall: 0.6286, Validation F1: 0.7527, Validation AUC-ROC: nan
Validation Confusion Matrix:
[[ 55   1 117]
 [  0 439   7]
 [  0  23 313]]
Model saved at model.pth
Epoch [2/10], Loss: 0.2277, Accuracy: 0.9234, Precision: 0.9240, Recall: 0.9186, F1: 0.9083, AUC-ROC: nan, Grad Norm: 1.9606, Weight Norm: 88.4499, Time: 127.73s
Confusion Matrix:
[[ 579    1  109]
 [   0 1735   47]
 [  74   61 1208]]
Validation Loss: 0.3694, Validation Accuracy: 0.8681, Validation Precision: 0.8688, Validation Recall: 0.7111, Validation F1: 0.7996, Validation AUC-ROC: nan
Validation Confusion Matrix:
[[ 75   1  97]
 [  0 441   5]
 [  6  17 313]]
Model saved at model.pth
Epoch [3/10], 

In [10]:
def predict_image(image_path, model, transforms, device):
    image = Image.open(image_path).convert('RGB')
    
    image = transforms(image).unsqueeze(0)
    
    image = image.to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(image)
        
        probabilities = F.softmax(output, dim=1)        
        _, predicted_class = torch.max(output, 1)
        predicted_label = class_names[predicted_class.item()]
        
        return predicted_label, probabilities.cpu().numpy()

In [13]:
class_names = ['above', 'below', 'focused']  # Assuming these are the 3 classes

model = setup_model()
model.load_state_dict(torch.load(model_path))

all_images = []
for class_name in class_names:
    class_dir = os.path.join(classified_images, class_name)
    image_files = [os.path.join(class_dir, img) for img in os.listdir(class_dir) if img.endswith(('.jpg'))]
    all_images.extend(image_files)

random_images = random.sample(all_images, 100)

# Inference
correct_predictions = 0
for image_path in random_images:
    actual_class = os.path.basename(os.path.dirname(image_path))
    predicted_label, _ = predict_image(image_path, model, val_transforms, device)
    if predicted_label == actual_class:
        correct_predictions += 1
    
    print(f'Image: {image_path}, Actual: {actual_class}, Predicted: {predicted_label}')

accuracy = correct_predictions / len(random_images)
print(f'Accuracy on 100 randomly selected images: {accuracy:.4f}')

Image: ../classified_images/above/34604_1725465246.315834.jpg, Actual: above, Predicted: above
Image: ../classified_images/below/14862_1726847719.392984.jpg, Actual: below, Predicted: below
Image: ../classified_images/focused/7559_1726847604.634238.jpg, Actual: focused, Predicted: focused
Image: ../classified_images/below/34819_1725465249.410111.jpg, Actual: below, Predicted: focused
Image: ../classified_images/focused/35271_1725465255.781661.jpg, Actual: focused, Predicted: focused
Image: ../classified_images/above/11022_1726847654.103875.jpg, Actual: above, Predicted: above
Image: ../classified_images/below/14807_1726847718.57274.jpg, Actual: below, Predicted: below
Image: ../classified_images/below/12715_1726847680.893261.jpg, Actual: below, Predicted: below
Image: ../classified_images/focused/8601_1726847618.659431.jpg, Actual: focused, Predicted: focused
Image: ../classified_images/below/12826_1726847682.771626.jpg, Actual: below, Predicted: below
Image: ../classified_images/above